# Cyclist Pain Points: Phase 2 OSM Prototype

This notebook is the first implementation step for the separate cyclist_painpoints project.
It only does the OSM side of the pipeline:

- lock one small study area by fixing a single bbox in the config cell
- pull the road network for that bbox from OpenStreetMap via OSMnx
- extract `highway` and `cycleway` tags per edge
- mark edges as `unprotected` when `highway` is primary/secondary/tertiary and `cycleway` is missing or `no`
- save GeoJSON, CSV, and a quick Folium map preview under `cyclist_painpoints/`

## What "lock" and "OSM pull" mean
- Lock the study area means: pick one bbox once and keep it fixed while you work
- OSM pull means: ask OSMnx to download the road network inside that bbox from OpenStreetMap
- This phase is CPU only; no GPU is needed
- This phase does not use orthophotos, SegEarth-OV3 inference, or segmentation fusion yet

## Section 1: Set Up Project Structure

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
import math

import folium
import geopandas as gpd
import osmnx as ox
import pandas as pd

# Use a mirror and skip rate-limit polling so the notebook queries Overpass directly.
ox.settings.overpass_url = "https://overpass.kumi.systems/api"
ox.settings.overpass_rate_limit = False
ox.settings.requests_timeout = 300
# requests_timeout is only the client-side HTTP timeout. The Overpass query itself
# carries its own server-side [timeout:N] clause, which OSMnx defaults to 180s
# regardless of requests_timeout. Raise it explicitly so a busy mirror gets the
# full budget before the query is killed server-side.
ox.settings.overpass_settings = "[out:json][timeout:300]{maxsize}"

In [ ]:
PROJECT_ROOT = next((parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "CLAUDE.md").exists()), Path.cwd().resolve())
CYCLIST_ROOT = PROJECT_ROOT / "cyclist_painpoints"
NOTEBOOK_DIR = CYCLIST_ROOT / "notebooks"
DATA_DIR = CYCLIST_ROOT / "data"
RESULTS_DIR = CYCLIST_ROOT / "results"

for directory in [NOTEBOOK_DIR, DATA_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

In [ ]:
ROAD_CLASSES = {"primary", "secondary", "tertiary"}
PROTECTED_CYCLEWAY_VALUES = {"lane", "track", "opposite_lane", "shared_lane", "share_busway", "separate"}


@dataclass(frozen=True)
class BBox:
    north: float
    south: float
    east: float
    west: float

    def as_osmnx(self) -> tuple[float, float, float, float]:
        # OSMnx expects (west, south, east, north).
        return (self.west, self.south, self.east, self.north)


@dataclass(frozen=True)
class EdgeTagSummary:
    u: int
    v: int
    key: int
    highway: str | None
    cycleway: str | None
    unprotected: bool


def normalize_tag_values(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, float) and math.isnan(value):
        return []
    values = value if isinstance(value, (list, tuple, set)) else [value]
    normalized: list[str] = []
    for item in values:
        if item is None:
            continue
        text = str(item).strip().lower()
        if text and text != "nan":
            normalized.append(text)
    return normalized


def first_tag_value(value) -> str | None:
    values = normalize_tag_values(value)
    return values[0] if values else None


def is_unprotected_edge(highway_value, cycleway_value) -> bool:
    highway_values = normalize_tag_values(highway_value)
    cycleway_values = [value for value in normalize_tag_values(cycleway_value) if value != "no"]
    has_target_highway = any(value in ROAD_CLASSES for value in highway_values)
    has_protection = any(value in PROTECTED_CYCLEWAY_VALUES for value in cycleway_values)
    return has_target_highway and not has_protection

## Section 4: OSM Fetch, Save, and Map Helpers

These are the reusable functions for this workflow: `fetch_osm_edges` pulls the
road graph from Overpass and tags each edge, `save_edge_outputs` writes the
GeoJSON and CSV, and `build_preview_map` renders the Folium HTML preview
(red = unprotected, blue = protected).

In [ ]:
def fetch_osm_edges(bbox: BBox, network_type: str = "drive") -> gpd.GeoDataFrame:
    graph = ox.graph_from_bbox(bbox.as_osmnx(), network_type=network_type, simplify=True, retain_all=False)
    _, edges = ox.graph_to_gdfs(graph, nodes=True, edges=True, fill_edge_geometry=True)
    edges = edges.reset_index()
    edges["highway_raw"] = edges.get("highway")
    edges["cycleway_raw"] = edges.get("cycleway")
    edges["highway"] = edges["highway_raw"].map(first_tag_value)
    edges["cycleway"] = edges["cycleway_raw"].map(first_tag_value)
    edges["unprotected"] = edges.apply(
        lambda row: is_unprotected_edge(row["highway_raw"], row["cycleway_raw"]), axis=1
    )
    return edges


def save_edge_outputs(edges: gpd.GeoDataFrame, data_dir: Path, results_dir: Path, stem: str) -> dict[str, Path]:
    geojson_path = data_dir / f"{stem}.geojson"
    csv_path = data_dir / f"{stem}.csv"
    html_path = results_dir / f"{stem}.html"

    edges.to_file(geojson_path, driver="GeoJSON")
    edges.drop(columns="geometry").to_csv(csv_path, index=False)
    return {"geojson": geojson_path, "csv": csv_path, "html": html_path}


def build_preview_map(edges: gpd.GeoDataFrame, bbox: BBox, html_path: Path) -> Path:
    preview = edges.to_crs(epsg=4326)
    center_lat = (bbox.north + bbox.south) / 2
    center_lon = (bbox.east + bbox.west) / 2
    fmap = folium.Map(location=[center_lat, center_lon], zoom_start=16, tiles="CartoDB positron")

    def edge_style(feature):
        return {
            "color": "#d73027" if feature["properties"].get("unprotected") else "#4575b4",
            "weight": 4,
            "opacity": 0.85,
        }

    folium.GeoJson(
        preview[["geometry", "highway", "cycleway", "unprotected"]],
        style_function=edge_style,
        tooltip=folium.GeoJsonTooltip(fields=["highway", "cycleway", "unprotected"]),
        name="road edges",
    ).add_to(fmap)
    folium.LayerControl().add_to(fmap)
    fmap.save(html_path)
    return html_path

## Section 5: Create a Minimal Runnable Example

In [ ]:
# Lock the study area here by changing only this one bbox.
# This is a larger sub-area inside the selected DOP20 tile footprint for dop20_32_473_5521_1_he.
# Keep it fixed while developing so your OSM output stays comparable.
STUDY_AREA_NAME = "dop20_32_473_5521_1_he_medium"
BBOX = BBox(
    north=49.84858479248858,
    south=49.842260278121785,
    east=8.664080086662173,
    west=8.654388311831408,
)

OUTPUT_STEM = STUDY_AREA_NAME

# OSM pull: this downloads the road graph inside the bbox from OpenStreetMap.
edges = fetch_osm_edges(BBOX, network_type="drive")
paths = save_edge_outputs(edges, DATA_DIR, RESULTS_DIR, OUTPUT_STEM)
map_path = build_preview_map(edges, BBOX, paths["html"])

print(f"study area: {STUDY_AREA_NAME}")
print(f"edges: {len(edges)}")
print("saved geojson:", paths["geojson"])
print("saved csv:", paths["csv"])
print("saved map:", map_path)
edges[["highway", "cycleway", "unprotected"]].head(10)

## Section 5b: Second Study Area with Arterial Roads

The Section 5 bbox is residential-only, so it has no unprotected edges to rank
(see Section 7). This second bbox is locked over central Darmstadt near
Rheinstrasse, which carries primary/secondary roads, so the pain-point ranking
in Sections 7-8 has real candidates to work with. `edges` and `paths` are
reassigned here and carried forward into Sections 6-8.

In [ ]:
# Second locked study area: central Darmstadt near Rheinstrasse (primary/secondary roads present).
STUDY_AREA_NAME = "darmstadt_rheinstrasse_arterial"
BBOX = BBox(
    north=49.8710,
    south=49.8680,
    east=8.6480,
    west=8.6430,
)

OUTPUT_STEM = STUDY_AREA_NAME

edges = fetch_osm_edges(BBOX, network_type="drive")
paths = save_edge_outputs(edges, DATA_DIR, RESULTS_DIR, OUTPUT_STEM)
map_path = build_preview_map(edges, BBOX, paths["html"])

print(f"study area: {STUDY_AREA_NAME}")
print(f"edges: {len(edges)}")
print("saved geojson:", paths["geojson"])
print("saved csv:", paths["csv"])
print("saved map:", map_path)
edges[["highway", "cycleway", "unprotected"]].value_counts(dropna=False)

## Section 6: Add Basic Validation Checks

In [ ]:
required_columns = {"highway", "cycleway", "unprotected", "geometry"}
missing_columns = required_columns - set(edges.columns)
assert not missing_columns, f"Missing expected columns: {missing_columns}"
assert edges["geometry"].notna().all(), "Found empty geometry values"
assert edges["unprotected"].isin([True, False]).all(), "unprotected must be boolean-like"
assert edges["highway"].notna().any(), "Expected at least one edge with a highway tag"

summary = edges["unprotected"].value_counts(dropna=False).to_dict()
print("validation summary:", summary)

## Section 7: Score and Rank Pain Points

OSM tags alone (no imagery, no segmentation) give a simple, defensible pain-point
proxy: an unprotected edge is riskier the busier its road class is and the longer
the unprotected stretch is. `PAIN_POINT_WEIGHTS` encodes that primary roads carry
more risk per metre than secondary, and secondary more than tertiary. The score is
`weight[highway] * length_m` for unprotected edges only; protected and non-target
edges score 0. This ranks *where* to look first, not a calibrated probability.

In [ ]:
PAIN_POINT_WEIGHTS = {"primary": 3.0, "secondary": 2.0, "tertiary": 1.0}


def score_pain_points(edges: gpd.GeoDataFrame, top_n: int = 10) -> gpd.GeoDataFrame:
    scored = edges.copy()
    weight = scored["highway"].map(PAIN_POINT_WEIGHTS).fillna(0.0)
    scored["pain_score"] = weight * scored["length"] * scored["unprotected"]
    ranked = scored.sort_values("pain_score", ascending=False)
    return ranked[ranked["pain_score"] > 0].head(top_n)


top_pain_points = score_pain_points(edges, top_n=10)
print(f"unprotected edges with a nonzero pain score: {len(top_pain_points)}")
if top_pain_points.empty:
    print("No primary/secondary/tertiary edges without cycleway protection in this bbox.")
    print("This bbox is residential-only by design (see Section 5) -- lock a different bbox to see ranked pain points.")
top_pain_points[["name", "highway", "cycleway", "length", "pain_score"]]

## Section 8: Export Ranked Pain Points for QGIS

Save the top-N ranked edges as their own GeoJSON so they can be loaded directly
into QGIS as a separate layer, styled by `pain_score`, and overlaid on the
orthophoto or the full road-edges layer from Section 5 for validation.

In [ ]:
def save_pain_points(top_pain_points: gpd.GeoDataFrame, data_dir: Path, stem: str) -> Path:
    pain_point_path = data_dir / f"{stem}_pain_points.geojson"
    export_columns = ["u", "v", "key", "osmid", "name", "highway", "cycleway", "length", "pain_score", "geometry"]
    top_pain_points[export_columns].to_file(pain_point_path, driver="GeoJSON")
    return pain_point_path


pain_point_path = save_pain_points(top_pain_points, DATA_DIR, OUTPUT_STEM)
print(f"top pain points: {len(top_pain_points)}")
print("saved pain points geojson:", pain_point_path)
print("QGIS: load this alongside", paths["geojson"].name, "and style by pain_score")